# Qwen2.5-1.5B · WinoGrande и MATH

In [ ]:
import hashlib
import json
import os
import zipfile
from pathlib import Path

project = "qwen25"
archive_name = "replaceme-qwen25-env.zip"
expected_manifest = "d38c6d7a45ea159aae1b75d24a1587d302cab8cac127c5845ce725c79400f086"
kaggle = Path("/kaggle/input").is_dir()
default_input = Path("/kaggle/input/replaceme-qwen25-env") if kaggle else Path.cwd()
inputs = Path(os.environ.get("REPLACEME_INPUT", default_input)).expanduser().resolve()
work = Path(os.environ.get("REPLACEME_WORK", "/kaggle/working" if kaggle else "replaceme-work")).resolve()

archive_path = inputs if inputs.name == archive_name else inputs / archive_name
if archive_path.is_file():
    bundle = archive_path
    with zipfile.ZipFile(bundle) as archive:
        metadata = archive.read("environment.json")
        bootstrap = archive.read("bootstrap.py")
elif inputs.is_dir() and (inputs / "environment.json").is_file():
    bundle = inputs
    metadata = (bundle / "environment.json").read_bytes()
    bootstrap = (bundle / "bootstrap.py").read_bytes()
else:
    raise FileNotFoundError(f"Ожидается {archive_path}. Укажите точный путь к ZIP или каталогу Dataset в REPLACEME_INPUT.")

if hashlib.sha256(metadata).hexdigest() != expected_manifest:
    raise ValueError(f"Манифест не соответствует комплекту {archive_name}.")
manifest = json.loads(metadata)
if manifest["project"] != project or hashlib.sha256(bootstrap).hexdigest() != manifest["bootstrap_sha256"]:
    raise ValueError("Ноутбук и загрузчик окружения относятся к разным сборкам.")
scope = {}
exec(compile(bootstrap, "bootstrap.py", "exec"), scope)
scope["start"](bundle, manifest, work / project)


## Модель и данные

In [ ]:
%%benchmark
import json
import time
from contextlib import closing
from pathlib import Path
from importlib.metadata import version

import numpy as np
from llama_cpp import Llama
from math_verify import parse, verify
from qwen25_data import load_benchmark, weights_path

weights = weights_path()
data = load_benchmark()
options = dict(model_path=str(weights), n_gpu_layers=-1, n_threads=4, n_batch=64, verbose=False)
out = Path.cwd() / "qwen25-results"
out.mkdir(exist_ok=True)

def save(name, rows):
    tmp = out / (name + ".tmp")
    tmp.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows))
    tmp.replace(out / (name + ".jsonl"))

print(f"{weights.name} · llama-cpp-python {version('llama-cpp-python')}")
print(f"WinoGrande {len(data['winogrande'])} · MATH {len(data['math'])}")

## WinoGrande · 5-shot

In [ ]:
%%benchmark
demos = "\n\n".join(r["sentence"].replace("_", r["option" + r["answer"]])
                     for r in data["winogrande_demos"]) + "\n\n"

def suffix_score(model, context, suffix):
    prefix = model.tokenize(context.encode(), add_bos=False)
    ids = model.tokenize((context + suffix).encode(), add_bos=False)
    assert ids[:len(prefix)] == prefix and len(prefix) < len(ids) <= model.n_ctx()
    model.n_tokens = model.longest_token_prefix(model.input_ids[:model.n_tokens], ids)
    model.eval(ids[model.n_tokens:])
    logits = model.scores[len(prefix)-1:len(ids)-1]
    targets = ids[len(prefix):]
    peaks = logits.max(axis=1)
    log_z = peaks + np.log(np.exp(logits - peaks[:, None]).sum(axis=1))
    return float((logits[np.arange(len(targets)), targets] - log_z).sum())

start = time.monotonic()
wino = []
with closing(Llama(**options, n_ctx=384, logits_all=True)) as model:
    for row in data["winogrande"]:
        left, right = row["sentence"].split("_")
        scores = [suffix_score(model, demos + left + row[key].rstrip(), " " + right.strip())
                  for key in ("option1", "option2")]
        pred = str(1 + int(np.argmax(scores)))
        wino.append(dict(index=row["index"], gold=row["answer"], pred=pred, scores=scores,
                         correct=pred == row["answer"]))
wino_seconds = time.monotonic() - start
save("winogrande", wino)
print(f"Верно {sum(r['correct'] for r in wino)} из {len(wino)}")

## MATH · 4-shot

In [ ]:
%%benchmark
def last_box(text):
    start = text.rfind(r"\boxed{")
    if start < 0:
        return None
    start += len(r"\boxed{")
    depth = 1
    for i in range(start, len(text)):
        depth += (text[i] == "{") - (text[i] == "}")
        if depth == 0:
            return text[start:i]

def equivalent(gold, pred):
    if pred is None:
        return False
    if gold.strip() == pred.strip():
        return True
    return bool(verify(parse(r"$\boxed{" + gold + "}$"), parse(r"$\boxed{" + pred + "}$")))

def problem(text):
    return "Problem:\n" + text + "\n\nSolution:\n"

examples = "\n\n".join(problem(q) + a for q, a in data["math_demos"]) + "\n\n"
math_rows = []
start = time.monotonic()
with closing(Llama(**options, n_ctx=3072)) as model:
    for row in data["math"]:
        prompt = examples + problem(row["problem"])
        ids = model.tokenize(prompt.encode(), add_bos=False)
        assert len(ids) + 768 <= model.n_ctx()
        answer = model(ids, max_tokens=768, temperature=0, repeat_penalty=1,
                       stop=["\nProblem:"], echo=False)["choices"][0]
        gold, pred = last_box(row["solution"]), last_box(answer["text"])
        assert gold is not None
        math_rows.append(dict(subject=row["subject"], level=row["level"], index=row["index"],
                              problem=row["problem"], completion=answer["text"],
                              gold=gold, pred=pred, correct=equivalent(gold, pred),
                              finish_reason=answer["finish_reason"]))
math_seconds = time.monotonic() - start
save("math", math_rows)
print(f"Верно {sum(r['correct'] for r in math_rows)} из {len(math_rows)}")

## Результаты

In [ ]:
%%benchmark
from IPython.display import Markdown, display

summary = {}
table = ["| Задача | Верно | Точность | Статья |", "|---|---:|---:|---:|"]
for name, rows, ref, shots, seconds in [
    ("WinoGrande", wino, 65.0, 5, wino_seconds), ("MATH", math_rows, 35.0, 4, math_seconds)
]:
    n = len(rows)
    hits = sum(r["correct"] for r in rows)
    acc = 100 * hits / n
    summary[name] = dict(correct=hits, total=n, accuracy_percent=acc, report_percent=ref,
                         shots=shots, seconds=round(seconds, 1))
    table.append(f"| {name} | {hits}/{n} | {acc:.1f}% | {ref:.1f}% |")
(out / "summary.json").write_text(json.dumps(dict(model=weights.name, seed=data["seed"],
    runtime=version("llama-cpp-python"), math_verify=version("math-verify"), results=summary), indent=2))
display(Markdown("\n".join(table)))
print("Ответы и метрики:", out)